In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import pandas as pd

In [2]:
p = {
    'kks_1':  0.95,   #QoI
    'kkr_1':  0,   #QoI
    'kk_2':  0.95,    #SDHI
    'C50s_1': 0.02,    #QoI_azoxystrobin
    'C50s_2': 0.06,  #SDHI_fluxapyroxad sensitive
    'C50r_2': 1.4,
    'K':     1.0,    
    'beta':  0.2,    
    'mu':    0.1,    
    'rH':    0.1,   
    'rhor_1': 0.02,   #QoI
    'rhor_2': 0.1,     #SDHI
    'm':    3.2e-5  
}

C = np.array([0, 0.25, 0.5, 1, 2, 4]) #dose strategies, 1 represents baseline dose. (0.25-1 is selection window)

In [3]:
y0 = [0.95,     
      0.03,   
      0.01,
      0.01,
      0.01]   
t = np.linspace(0, 2000, 2000)

In [4]:
def Hills_function (C, kk, C50):
   return (kk * C / (C + C50))

C1, C2 = np.meshgrid(C, C)

e_1 = Hills_function(C1,p['kks_1'],p['C50s_1']) #QoI_azoxystrobin sensitive
es_2 = Hills_function(C2,p['kk_2'],p['C50s_2']) #SDHI_fluxapyroxad sensitive
er_2 = Hills_function(C2,p['kk_2'],p['C50r_2']) #SDHI_fluxapyroxad resistant

b_ab = p['beta']*(1-e_1)*(1-es_2)
b_Ab = p['beta']*(1-p['rhor_1'])*(1-es_2)
b_aB = p['beta']*(1-e_1)*(1-p['rhor_2'])*(1-er_2)  #SDHI resistance is quantitative, while QoI can be assumed to be qualitative
b_AB = p['beta']*(1-p['rhor_1'])*(1-p['rhor_2'])*(1-er_2)


df_doses = pd.DataFrame ({
    'QoI': C1.flatten(),
    'SDHI' : C2.flatten(),
    'b_ab': b_ab.flatten(),
    'b_Ab': b_Ab.flatten(),
    'b_aB': b_aB.flatten(),
    'b_AB': b_AB.flatten(),
})
print( 'fungicide mixtures and transmission rates')
df_doses   

fungicide mixtures and transmission rates


,QoI,SDHI,b_ab,b_Ab,b_aB,b_AB
0,0.00,0.00,0.200000,0.196000,0.180000,0.176400
1,0.25,0.00,0.024074,0.196000,0.021667,0.176400
2,0.50,0.00,0.017308,0.196000,0.015577,0.176400
3,1.00,0.00,0.013725,0.196000,0.012353,0.176400
4,2.00,0.00,0.011881,0.196000,0.010693,0.176400
5,4.00,0.00,0.010945,0.196000,0.009851,0.176400
6,0.00,0.25,0.046774,0.045839,0.154091,0.151009
7,0.25,0.25,0.005630,0.045839,0.018548,0.151009
8,0.50,0.25,0.004048,0.045839,0.013335,0.151009
9,1.00,0.25,0.003210,0.045839,0.010575,0.151009


In [5]:
def ODEs (y,t,p,b_ab, b_Ab, b_aB, b_AB):
    H, I_ab, I_Ab, I_aB, I_AB = y
    m = p['m']
    rH = p ['rH']
    K = p['K']
    mu = p['mu']

    dH = rH*(K - H - I_ab - I_Ab - I_aB - I_AB) - b_ab*I_ab*H - b_Ab*I_Ab*H - b_aB*I_aB*H - b_AB*I_AB*H
    dI_ab = (1-2*m)*b_ab*I_ab*H + m*b_Ab*I_Ab*H + m*b_aB*I_aB*H - mu*I_ab
    dI_Ab = m*b_ab*I_ab*H + (1-2*m)*b_Ab*I_Ab*H + m*b_AB*I_AB*H - mu*I_Ab
    dI_aB = m*b_ab*I_ab*H + (1-2*m)*b_aB*I_aB*H + m*b_AB*I_AB*H - mu*I_aB
    dI_AB = m*b_Ab*I_Ab*H + m*b_aB*I_aB*H + (1-2*m)*b_AB*I_AB*H - mu*I_AB

    return [dH, dI_ab, dI_Ab, dI_aB, dI_AB]

In [8]:
def calculate_90_threshold(sol, t):
    I_total = sol[:,1] + sol[:,2] + sol[:,3] + sol[:,4]
    I_AB = sol[:,4]

    valid = np.isfinite(I_total) & np.isfinite(I_AB)

    I_total_safe = np.where(valid, I_total, 0.0)
    I_AB_safe = np.where(valid, I_AB, 0.0)

    proportion = np.zeros_like(I_total_safe)

    mask = I_total_safe > 1e-8
    proportion[mask] = I_AB_safe[mask] / I_total_safe[mask]

    indices = np.where(proportion >= 0.9)[0]

    if len(indices) > 0:
        return t[indices[0]]
    else:
        return np.nan

In [9]:
results = []

for i in range(len(C)):
    for j in range(len(C)):
        sol = odeint(
            ODEs,
            y0,
            t,
            args=(p, b_ab[i,j], b_Ab[i,j], b_aB[i,j], b_AB[i,j])
        )
        sol = np.maximum(sol, 0)
       
        eq = np.mean(sol[-100:], axis=0)
        
        H_eq, I_ab_eq, I_Ab_eq, I_aB_eq, I_AB_eq = eq
        I_total_eq = I_ab_eq + I_Ab_eq + I_aB_eq + I_AB_eq
        if I_total_eq > 1e-8:
            I_AB_prop = I_AB_eq / I_total_eq
        else:
            I_AB_prop = 0.0

        t90 = calculate_90_threshold(sol, t)

        results.append({
            "QoI": C[i],
            "SDHI": C[j],

            "$H^*$": H_eq,
            "$I_{ab}^*$": I_ab_eq,
            "$I_{Ab}^*$": I_Ab_eq,
            "$I_{aB}^*$": I_aB_eq,
            "$I_{AB}^*$": I_AB_eq,
            "t90": t90,
            
        })

df_results = pd.DataFrame(results)
df_results

,QoI,SDHI,$H^*$,$I_{ab}^*$,$I_{Ab}^*$,$I_{aB}^*$,$I_{AB}^*$,t90
0,0.00,0.00,0.500097,2.478790e-01,1.992881e-03,7.932821e-05,6.359801e-07,NaN
1,0.00,0.25,0.510237,8.930872e-06,2.447944e-01,2.576211e-09,7.833945e-05,NaN
2,0.00,0.50,0.510237,8.592707e-06,2.447947e-01,2.477528e-09,7.834025e-05,NaN
3,0.00,1.00,0.510237,8.423842e-06,2.447949e-01,2.428262e-09,7.834012e-05,NaN
4,0.00,2.00,0.510237,8.339464e-06,2.447949e-01,2.403783e-09,7.834386e-05,NaN
5,0.00,4.00,0.510237,8.297289e-06,2.447950e-01,2.391370e-09,7.834048e-05,NaN
6,0.25,0.00,0.649260,7.894736e-06,1.629734e-07,1.718108e-01,3.552191e-03,NaN
7,0.25,0.25,0.662254,1.034456e-10,7.759105e-06,6.160510e-06,1.688590e-01,32.016008
8,0.25,0.50,0.662254,9.466093e-11,7.759116e-06,5.927244e-06,1.688592e-01,31.015508
9,0.25,1.00,0.662254,9.031564e-11,7.759121e-06,5.810761e-06,1.688593e-01,31.015508
